# P9 — Validation avant AWS

Images → ResNet50 → centrage → PCA Spark → Parquet. Exécuter les cellules dans l’ordre. Les résultats ci-dessous seront ceux de cette nouvelle exécution, pas les anciens résultats. Aucun service AWS n’est créé par ce notebook.

In [ ]:
# Récupère la branche corrigée dans un dossier distinct pour préserver les anciennes copies.
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
REPO_URL = 'https://github.com/neutrinoox/projet9-big-data-fruits.git'
BRANCH = 'fix/p9-pre-aws-validation'
PROJECT_ROOT = Path('/content/p9-pre-aws-v2')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=PROJECT_ROOT, text=True).strip()
    if branch != BRANCH:
        raise RuntimeError('Cette copie utilise une autre branche : choisir un autre PROJECT_ROOT.')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
os.chdir(PROJECT_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit exécuté :', COMMIT)


In [ ]:
# Installe les versions du projet ; les calculs se feront dans de nouveaux processus Python.
# Cela évite de réutiliser une ancienne version TensorFlow déjà importée dans le kernel Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run(['java', '-version'], check=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EVIDENCE = PROJECT_ROOT / 'outputs' / ('evidence-' + RUN_ID)
EVIDENCE.mkdir(parents=True)
(EVIDENCE / 'commit.txt').write_text(COMMIT)
(EVIDENCE / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
# Lance les tests de protection des données avant de télécharger ou copier le dataset.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-v'], check=True)


## Données : une seule variante et le split Training
Le test télécharge 100 images officielles de Fruits-360, vérifiées par empreintes et fixées à un commit. Kaggle reste disponible pour augmenter le volume ; mettre USE_PINNED_SAMPLE à False pour télécharger le dataset complet.

In [ ]:
# Télécharge les 100 images officielles fixées, ou le dataset Kaggle complet sur demande.
USE_PINNED_SAMPLE = True
if USE_PINNED_SAMPLE:
    SOURCE_ROOT = PROJECT_ROOT / 'data' / 'fruits-validation'
    if not SOURCE_ROOT.exists():
        subprocess.run([sys.executable, '-m', 'scripts.download_validation_sample'], check=True)
    training_candidates = [SOURCE_ROOT / 'Training']
else:
    subprocess.run([sys.executable, '-m', 'scripts.download_dataset'], check=True)
    training_candidates = sorted(p for p in (PROJECT_ROOT / 'data' / 'fruits').rglob('Training') if p.is_dir())
print('Dossiers Training disponibles :')
for index, path in enumerate(training_candidates):
    print(index, path)


In [ ]:
# Retient automatiquement la variante 100×100 si elle est unique ; sinon renseigner TRAINING_DIR.
TRAINING_DIR = None  # Exemple : Path('/content/p9-pre-aws-v2/data/fruits/.../Training')
preferred = [p for p in training_candidates if 'fruits-360_100x100' in p.parts]
choices = preferred if preferred else training_candidates
if TRAINING_DIR is None:
    if len(choices) != 1:
        raise ValueError('Plusieurs variantes possibles : renseigner TRAINING_DIR avec un chemin affiché ci-dessus.')
    TRAINING_DIR = choices[0]
TRAINING_DIR = Path(TRAINING_DIR).resolve()
assert TRAINING_DIR.is_dir() and TRAINING_DIR.name == 'Training'
SAMPLE = PROJECT_ROOT / 'data' / 'sample-v2'
subprocess.run([sys.executable, '-m', 'scripts.prepare_sample', '--input', str(TRAINING_DIR),
                '--output', str(SAMPLE), '--images', '100', '--classes', '10'], check=True)
manifest = json.loads((SAMPLE / '_sample_manifest.json').read_text())
assert manifest['actual_images'] == 100 and manifest['classes'] == 10, 'Échantillon incomplet : contrôler les données.'
(EVIDENCE / 'sample_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Échantillon vérifié :', manifest['actual_images'], 'images,', manifest['classes'], 'classes')


## Prototype hors Spark
Cette étape confirme le comportement ResNet50/sklearn. Sa variance ne doit pas être présentée comme un résultat AWS.

In [ ]:
# Exécute le prototype dans un processus neuf et conserve ses résultats et son journal.
ENV = os.environ.copy()
ENV.update({'OMP_NUM_THREADS':'1', 'OPENBLAS_NUM_THREADS':'1', 'TF_NUM_INTRAOP_THREADS':'1',
            'TF_NUM_INTEROP_THREADS':'1', 'PYSPARK_PYTHON':sys.executable})
LOCAL_OUTPUT = EVIDENCE / 'local_features_pca.parquet'
with (EVIDENCE / 'local.log').open('w') as log:
    run = subprocess.run([sys.executable, '-m', 'src.pipeline_local', '--input', str(SAMPLE),
                          '--output', str(LOCAL_OUTPUT), '--max-images', '100', '--components', '20',
                          '--batch-size', '8'], env=ENV, stdout=log, stderr=subprocess.STDOUT)
print((EVIDENCE / 'local.log').read_text()[-8000:])
run.check_returncode()


## Pipeline Spark complet
Deux partitions actives et deux threads locaux permettent de tester le découpage. La preuve de plusieurs machines sera réalisée sur EMR.

In [ ]:
# Construit l’archive distribuée aux workers et exécute le pipeline avec deux threads locaux.
subprocess.run([sys.executable, '-m', 'scripts.package_emr'], check=True)
SPARK_OUTPUT = PROJECT_ROOT / 'outputs' / ('spark-' + RUN_ID)
with (EVIDENCE / 'spark.log').open('w') as log:
    run = subprocess.run(['spark-submit', '--master', 'local[2]', '--driver-memory', '4g',
                          '--py-files', 'dist/p9_src.zip', 'dist/run_pipeline.py',
                          '--input', str(SAMPLE), '--output', str(SPARK_OUTPUT), '--max-images', '100',
                          '--components', '20', '--batch-size', '8', '--partitions', '2'],
                         env=ENV, stdout=log, stderr=subprocess.STDOUT)
print((EVIDENCE / 'spark.log').read_text()[-12000:])
run.check_returncode()


In [ ]:
# Lit les métriques écrites après validation réelle du Parquet ; contrôle les résultats attendus.
metrics_files = list((SPARK_OUTPUT / 'metrics').glob('part-*'))
assert (SPARK_OUTPUT / 'metrics' / '_SUCCESS').exists() and metrics_files
metrics = json.loads(metrics_files[0].read_text())
assert metrics['status'] == 'validated' and metrics['images'] == 100
assert metrics['components'] == 20 and metrics['active_partitions'] >= 2
print(json.dumps(metrics, ensure_ascii=False, indent=2))


In [ ]:
# Visualise la variance cumulée réellement obtenue pour justifier le choix de k.
import matplotlib.pyplot as plt
cumulative = []
for value in metrics['variance_by_component']:
    cumulative.append(value + (cumulative[-1] if cumulative else 0))
plt.plot(range(1, len(cumulative)+1), cumulative, marker='o')
plt.axhline(0.90, linestyle='--', color='red', label='Repère à 90 %')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance expliquée cumulée')
plt.title('PCA Spark — ce run local')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(EVIDENCE / 'variance_spark.png', bbox_inches='tight')
plt.show()


In [ ]:
# Archive le run, le manifeste du code et les journaux avant la fermeture du runtime Colab.
import shutil
shutil.copy2(PROJECT_ROOT / 'dist' / 'manifest.json', EVIDENCE / 'code_manifest.json')
shutil.copytree(SPARK_OUTPUT, EVIDENCE / 'spark_run')
ARCHIVE = shutil.make_archive(str(EVIDENCE), 'zip', EVIDENCE)
print('À conserver avant de quitter Colab :', ARCHIVE)
# Déclenche le téléchargement des preuves si le notebook est exécuté dans Colab.
try:
    from google.colab import files
except ImportError:
    files = None
if files is not None:
    files.download(ARCHIVE)


## Étape suivante : AWS
Après réussite de ce notebook, suivre `docs/aws_plan.md`, `docs/commands.md` et `notebooks/P9_execution_EMR.ipynb`. La région, les accès IAM, le budget et la configuration du cluster seront validés avant de créer les ressources. Conserver aussi ce notebook exécuté. Ne pas annoncer les CE cloud acquis à ce stade.